# Stage 2: CKIP 任務別集成與後處理

讀取 Stage 1 機率檔，訓練 CKIP 任務別模型，搜尋後處理與任務別 ensemble 設定。

預設 Drive root：`/content/drive/MyDrive/VeriPromiseESG2026`。


## 0. Colab 執行環境與路徑檢查

請先執行下一個 setup cell。它會安裝套件、掛載 Google Drive、檢查 GPU、建立輸出資料夾，並驗證必要輸入檔是否存在。


In [ ]:
# Colab 啟動區塊：安裝套件、掛載 Drive、檢查 GPU、驗證必要輸入檔。
!pip -q install transformers accelerate scikit-learn pandas numpy tqdm openpyxl

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"
STAGE_NAME = "stage2_ckip_taskwise_ensemble"
STAGE_DISPLAY_NAME = "Stage 2 - CKIP 任務別集成"
OUTPUT_DIR = f"{OUTPUT_ROOT}/{STAGE_NAME}"
OUT_DIR = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

REQUIRED_RELATIVE_PATHS = [
    "data/vpesg4k_train_1000.json",
    "data/vpesg4k_valdata_1000.json",
    "data/vpesg4k_testdata_2000.json",
    "outputs/stage1_macbert_multitask_baseline/stage1_oof_val_test_probs.pkl",
]

def stage_log(label, value):
    print(f"[{STAGE_NAME}] {label}: {value}")

def require_files(relative_paths):
    missing = []
    for rel in relative_paths:
        path = f"{BASE_DIR}/{rel}"
        if not os.path.exists(path):
            missing.append(path)
    if missing:
        msg = "缺少必要輸入檔。請先執行前一個 Stage，或將資料放到 Google Drive：\n" + "\n".join(missing)
        raise FileNotFoundError(msg)

require_files(REQUIRED_RELATIVE_PATHS)
stage_log("Stage", STAGE_DISPLAY_NAME)
stage_log("BASE_DIR", BASE_DIR)
stage_log("DATA_DIR", DATA_DIR)
stage_log("OUTPUT_DIR", OUTPUT_DIR)

# 完整訓練建議使用 Colab A100 或同級 GPU。
try:
    gpu_names = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_names = list(gpu_names)
    stage_log("GPU", gpu_names)
    if not any("A100" in str(name) for name in gpu_names):
        stage_log("WARNING", "建議使用 A100；非 A100 runtime 可能需要更長時間。")
except Exception as exc:
    stage_log("WARNING", f"無法取得 GPU 資訊，請確認 Colab runtime 已啟用 GPU。{exc}")


## 輸入輸出契約

Stage 顯示名稱：`Stage 2 - CKIP 任務別集成`

Google Drive 輸出資料夾：

```text
/content/drive/MyDrive/VeriPromiseESG2026/outputs/stage2_ckip_taskwise_ensemble/
```

必要輸入檔：

- `data/vpesg4k_train_1000.json`
- `data/vpesg4k_valdata_1000.json`
- `data/vpesg4k_testdata_2000.json`
- `outputs/stage1_macbert_multitask_baseline/stage1_oof_val_test_probs.pkl`

主要輸出檔：

- `stage2_postprocess_best_config.json`
- `stage2_ckip_oof_val_test_probs.pkl`
- `stage2_best_val_config.json`
- `stage2_best_val_test2000_submission.csv`
- `stage2_balanced_test2000_submission.csv`

若必要輸入不存在，第一個 setup cell 會直接列出缺少的路徑並停止。


## Stage 主要流程

本節先重建 Stage 1 後處理版本，再加入 CKIP 任務別模型進行 ensemble。


In [ ]:

import os, json, pickle, re, math, itertools, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V30_DIR = f"{BASE_DIR}/outputs/stage1_macbert_multitask_baseline"
PROB_PATH = f"{V30_DIR}/stage1_oof_val_test_probs.pkl"

OUT_DIR = f"{BASE_DIR}/outputs/stage2_ckip_taskwise_ensemble"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}
LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

print("PROB_PATH:", PROB_PATH)
print("OUT_DIR:", OUT_DIR)


In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(PROB_PATH, "rb") as f:
    prob_obj = pickle.load(f)

val_probs = prob_obj["val_probs"]
test_probs = prob_obj["test_probs"]
oof_probs = prob_obj["oof_probs"]

print("train", train_df.shape)
print("val", val_df.shape)
print("test", test_df.shape)
print("loaded probs keys:", prob_obj.keys())
for t in TASKS:
    print(t, val_probs[t].shape, test_probs[t].shape)


In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    # 可選：如果 evidence_status=Yes，但 quality 被預測成 N/A，就改選非 N/A 中機率最高者
    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            # 禁止 N/A
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out


In [ ]:

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}
    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    # T1
    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    # T2
    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    # T3
    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                # 在 N/A / No 中挑一個較高者
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    # T4
    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred

def eval_config(config, verbose=False, title=""):
    pred = probs_to_pred_df(val_df, val_probs, config=config)
    m = calc_metrics(val_df, pred)
    if verbose:
        print_metrics(m, title or "config")
        for t in TASKS:
            print(t, pred[t].value_counts().to_dict())
    return m["weighted_score"], m, pred

base_config = {}
base_score, base_metrics, base_pred = eval_config(base_config, verbose=True, title="baseline from v30 probs")


In [ ]:

# 先試一個最常見的修正：降低 Not Clear、提高 Clear
manual_configs = [
    {
        "name": "manual_t4_less_notclear",
        "multipliers": {
            "evidence_quality": {
                "N/A": 1.0,
                "Clear": 2.0,
                "Not Clear": 0.35,
                "Misleading": 1.0,
            }
        },
        "force_quality_non_na_when_evidence_yes": True,
    },
    {
        "name": "manual_t4_clear_strong",
        "multipliers": {
            "evidence_quality": {
                "N/A": 1.0,
                "Clear": 3.0,
                "Not Clear": 0.25,
                "Misleading": 1.0,
            }
        },
        "force_quality_non_na_when_evidence_yes": True,
    },
]

for cfg in manual_configs:
    name = cfg.pop("name")
    score, m, pred = eval_config(cfg, verbose=True, title=name)
    cfg["name"] = name


In [ ]:

# Staged search：先搜 T4 class multipliers，因為目前最大問題在 evidence_quality 分布
def copy_config(cfg):
    return json.loads(json.dumps(cfg))

def coordinate_search(base_config, search_space, rounds=3):
    best_cfg = copy_config(base_config)
    best_score, best_metrics, best_pred = eval_config(best_cfg)
    history = []

    for r in range(rounds):
        improved = False
        print(f"\n--- coordinate round {r+1} start score {best_score:.6f} ---")
        for key, values in search_space:
            local_best = (best_score, copy_config(best_cfg), best_metrics)
            for val in values:
                cfg = copy_config(best_cfg)

                # key can be tuple path
                if key[0] == "mult":
                    _, task, label = key
                    cfg.setdefault("multipliers", {}).setdefault(task, {})[label] = val
                elif key[0] == "flag":
                    _, name = key
                    cfg[name] = val
                elif key[0] == "threshold":
                    _, name = key
                    cfg[name] = val

                score, metrics, _ = eval_config(cfg)
                if score > local_best[0]:
                    local_best = (score, cfg, metrics)

            if local_best[0] > best_score + 1e-9:
                best_score, best_cfg, best_metrics = local_best
                improved = True
                print("improved", key, "->", best_score)
                print(best_cfg)

        history.append({"round": r+1, "score": best_score, "config": copy_config(best_cfg)})
        if not improved:
            break

    best_score, best_metrics, best_pred = eval_config(best_cfg)
    return best_cfg, best_score, best_metrics, best_pred, history

mult_values = [0.10, 0.20, 0.35, 0.50, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0, 6.0]
threshold_values = [round(x, 2) for x in np.arange(0.30, 0.81, 0.02)]

search_space = []

# T4 first
for lab in LABELS["evidence_quality"]:
    search_space.append((("mult", "evidence_quality", lab), mult_values))
search_space.append((("flag", "force_quality_non_na_when_evidence_yes"), [False, True]))

# T2 timeline
for lab in LABELS["verification_timeline"]:
    search_space.append((("mult", "verification_timeline", lab), mult_values))

# T1 / T3 threshold
search_space.append((("threshold", "t1_threshold"), [None] + threshold_values))
search_space.append((("threshold", "t3_threshold"), [None] + threshold_values))

best_cfg, best_score, best_metrics, best_pred, history = coordinate_search(
    base_config,
    search_space,
    rounds=4,
)

print_metrics(best_metrics, "BEST calibrated val1000")
print("best config:")
print(json.dumps(best_cfg, ensure_ascii=False, indent=2))

for t in TASKS:
    print("\n", t)
    print(best_pred[t].value_counts().to_dict())


In [ ]:

# 儲存 val1000 結果
best_val_pred = best_pred.copy()
best_val_full = pd.concat(
    [
        val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
        val_df[TASKS].add_prefix("true_").reset_index(drop=True),
        best_val_pred[TASKS].add_prefix("pred_").reset_index(drop=True),
    ],
    axis=1,
)

for t in TASKS:
    best_val_full[f"err_{t}"] = best_val_full[f"true_{t}"] != best_val_full[f"pred_{t}"]
best_val_full["any_error"] = best_val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

best_val_full.to_csv(f"{OUT_DIR}/stage2_postprocess_val1000_predictions.csv", index=False, encoding="utf-8-sig")
best_val_full.to_excel(f"{OUT_DIR}/v30_01_val1000_calibrated_error_analysis.xlsx", index=False)

# confusion
with pd.ExcelWriter(f"{OUT_DIR}/v30_01_val1000_calibrated_confusion.xlsx", engine="openpyxl") as writer:
    for t in TASKS:
        cm = confusion_matrix(val_df[t], best_val_pred[t], labels=LABELS[t])
        cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
        cm_df.to_excel(writer, sheet_name=t[:31])

with open(f"{OUT_DIR}/stage2_postprocess_best_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_config": best_cfg,
        "best_metrics": best_metrics,
        "baseline_metrics": base_metrics,
        "history": history,
    }, f, ensure_ascii=False, indent=2)

print("saved val outputs to", OUT_DIR)
display(best_val_full["any_error"].value_counts())
display(best_val_full.head())


In [ ]:

# 用 best config 產生 test2000 submission
test_pred = probs_to_pred_df(test_df, test_probs, config=best_cfg)

# exact-match override：test 段落若和 train 完全相同，使用 train 標籤
def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

override_count = 0
for i, row in test_df.iterrows():
    key = compact_text(row["data"])
    if key in train_map:
        for t in TASKS:
            test_pred.loc[i, t] = train_map[key][t]
        override_count += 1

test_pred = apply_hierarchy_to_pred_df(
    test_pred,
    force_quality_non_na_when_evidence_yes=best_cfg.get("force_quality_non_na_when_evidence_yes", False),
    probs=adjust_probs(test_probs, best_cfg.get("multipliers", {})),
)

sub = test_pred[["id"] + TASKS].copy()
sub_path = f"{OUT_DIR}/stage2_postprocess_test2000_submission.csv"
sub.to_csv(sub_path, index=False, encoding="utf-8-sig")

print("saved:", sub_path)
print("exact-match override count:", override_count)
for t in TASKS:
    print("\n", t)
    print(sub[t].value_counts().to_dict())

display(sub.head())


In [ ]:

# 報告
report = []
report.append("# Stage 2A val1000 postprocess calibration report")
report.append("")
report.append("## Baseline metrics")
for k, v in base_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("")
report.append("## Best calibrated metrics")
for k, v in best_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("")
report.append("## Best config")
report.append("```json")
report.append(json.dumps(best_cfg, ensure_ascii=False, indent=2))
report.append("```")
report.append("")
report.append("## Test distribution")
for t in TASKS:
    report.append(f"### {t}")
    for lab, cnt in sub[t].value_counts().to_dict().items():
        report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)
with open(f"{OUT_DIR}/stage2_postprocess_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:4000])
print("saved report:", f"{OUT_DIR}/stage2_postprocess_report.md")


## Stage 2B CKIP task-wise ensemble

This section uses public Stage paths and artifact names under `/content/drive/MyDrive/VeriPromiseESG2026`.


## 1. 設定路徑與基本參數

In [ ]:

import os
import json
import math
import pickle
import random
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V30_DIR = f"{BASE_DIR}/outputs/stage1_macbert_multitask_baseline"
MACBERT_PROB_PATH = f"{V30_DIR}/stage1_oof_val_test_probs.pkl"

V30_01_DIR = f"{BASE_DIR}/outputs/stage2_ckip_taskwise_ensemble"
V30_01_CONFIG_PATH = f"{V30_01_DIR}/stage2_postprocess_best_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage2_ckip_taskwise_ensemble"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

MODEL_NAME = "ckiplab/bert-base-chinese"
SEED = 43
N_SPLITS = 5
MAX_LEN = 384
EPOCHS = 4
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2
NUM_WORKERS = 0

USE_CLASS_WEIGHTS_BY_TASK = {
    "promise_status": True,
    "verification_timeline": True,
    "evidence_status": True,
    "evidence_quality": False,
}
CLASS_WEIGHT_MAX = 5.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)


In [ ]:

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


## 2. 讀取資料與 v30 機率

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(MACBERT_PROB_PATH, "rb") as f:
    mac_obj = pickle.load(f)

mac_val_probs = mac_obj["val_probs"]
mac_test_probs = mac_obj["test_probs"]

if os.path.exists(V30_01_CONFIG_PATH):
    with open(V30_01_CONFIG_PATH, "r", encoding="utf-8") as f:
        v30_01_obj = json.load(f)
    V30_01_BEST_CONFIG = v30_01_obj.get("best_config", {})
else:
    V30_01_BEST_CONFIG = {}

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("MacBERT probs:", MACBERT_PROB_PATH)
print("v30_01 config:")
print(json.dumps(V30_01_BEST_CONFIG, ensure_ascii=False, indent=2))


## 3. 評分、後處理、轉預測工具

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred


## 4. CKIP 模型與訓練函式

In [ ]:

class ESGDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row["data"]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors=None,
        )

        item = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)

        if self.has_labels:
            item["labels"] = {
                t: torch.tensor(LABEL2ID[t][row[t]], dtype=torch.long)
                for t in TASKS
            }
        return item

class MultiTaskClassifier(nn.Module):
    def __init__(self, model_name, label_dims, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        pooled_dim = hidden * 2
        self.dropouts = nn.ModuleDict({t: nn.Dropout(dropout) for t in TASKS})
        self.classifiers = nn.ModuleDict({t: nn.Linear(pooled_dim, label_dims[t]) for t in TASKS})

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        last_hidden = out.last_hidden_state
        cls_vec = last_hidden[:, 0, :]
        mask = attention_mask.unsqueeze(-1).float()
        mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        pooled = torch.cat([cls_vec, mean_vec], dim=-1)

        return {
            t: self.classifiers[t](self.dropouts[t](pooled))
            for t in TASKS
        }

def make_class_weights(fold_train_df):
    weights = {}
    for t in TASKS:
        if not USE_CLASS_WEIGHTS_BY_TASK.get(t, False):
            weights[t] = None
            continue
        counts = fold_train_df[t].value_counts().to_dict()
        total = len(fold_train_df)
        n_classes = len(LABELS[t])
        arr = []
        for lab in LABELS[t]:
            c = max(counts.get(lab, 0), 1)
            arr.append(total / (n_classes * c))
        w = torch.tensor(arr, dtype=torch.float)
        w = torch.clamp(w, max=CLASS_WEIGHT_MAX)
        weights[t] = w.to(DEVICE)
    return weights

def make_losses(fold_train_df):
    class_weights = make_class_weights(fold_train_df)
    return {t: nn.CrossEntropyLoss(weight=class_weights[t]) for t in TASKS}

def batch_to_device(batch):
    out = {}
    for k, v in batch.items():
        if k == "labels":
            out[k] = {t: y.to(DEVICE) for t, y in v.items()}
        else:
            out[k] = v.to(DEVICE)
    return out

@torch.no_grad()
def predict_probs(model, df, tokenizer, batch_size=16):
    model.eval()
    ds = ESGDataset(df, tokenizer, max_len=MAX_LEN, has_labels=False)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    all_probs = {t: [] for t in TASKS}

    for batch in tqdm(dl, desc="predict", leave=False):
        batch = batch_to_device(batch)
        logits = model(**batch)
        for t in TASKS:
            all_probs[t].append(torch.softmax(logits[t], dim=-1).cpu().numpy())

    return {t: np.concatenate(all_probs[t], axis=0) for t in TASKS}


In [ ]:

def save_state_dict_atomic(model, path):
    tmp_path = path + ".tmp"
    torch.save(model.state_dict(), tmp_path)
    if (not os.path.exists(tmp_path)) or os.path.getsize(tmp_path) < 1024:
        raise IOError(f"Checkpoint write failed or produced a suspiciously small file: {tmp_path}")
    os.replace(tmp_path, path)

def load_state_dict_checked(path):
    if (not os.path.exists(path)) or os.path.getsize(path) < 1024:
        raise IOError(f"Checkpoint is missing or suspiciously small: {path}")
    return torch.load(path, map_location=DEVICE)

def train_one_fold(fold, fold_train_df, fold_valid_df, tokenizer):
    print(f"\n========== CKIP Fold {fold} ==========")
    print("train:", fold_train_df.shape, "valid:", fold_valid_df.shape)

    model = MultiTaskClassifier(
        MODEL_NAME,
        label_dims={t: len(LABELS[t]) for t in TASKS},
        dropout=0.2,
    ).to(DEVICE)

    train_ds = ESGDataset(fold_train_df, tokenizer, max_len=MAX_LEN, has_labels=True)
    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = math.ceil(len(train_dl) / GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    losses = make_losses(fold_train_df)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    best_score = -1
    best_path = f"{OUT_DIR}/ckip_fold{fold}_best.pt"
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_dl, desc=f"ckip fold{fold} epoch{epoch}", leave=False)
        for step, batch in enumerate(pbar, start=1):
            batch = batch_to_device(batch)
            labels = batch.pop("labels")

            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                logits = model(**batch)
                loss = 0
                for t in TASKS:
                    loss = loss + losses[t](logits[t], labels[t])
                loss = loss / len(TASKS)
                loss = loss / GRAD_ACCUM

            scaler.scale(loss).backward()
            total_loss += loss.item() * GRAD_ACCUM

            if step % GRAD_ACCUM == 0 or step == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=total_loss / step)

        valid_probs = predict_probs(model, fold_valid_df, tokenizer, batch_size=16)
        valid_pred = probs_to_pred_df(fold_valid_df, valid_probs, config={})
        metrics = calc_metrics(fold_valid_df, valid_pred)
        score = metrics["weighted_score"]
        print_metrics(metrics, f"CKIP fold {fold} epoch {epoch} valid raw")

        if score > best_score:
            best_score = score
            bad_epochs = 0
            save_state_dict_atomic(model, best_path)
            print(f"saved best CKIP fold{fold}: {best_score:.6f}")
        else:
            bad_epochs += 1
            print("bad_epochs:", bad_epochs)
            if bad_epochs >= PATIENCE:
                print("early stopping")
                break

    model.load_state_dict(load_state_dict_checked(best_path))
    return model, best_score, best_path


## 5. 訓練 CKIP 5-fold stem

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

strat_key = train_df["promise_status"].astype(str) + "|" + train_df["evidence_status"].astype(str)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

ckip_oof_probs = {t: np.zeros((len(train_df), len(LABELS[t])), dtype=np.float32) for t in TASKS}
ckip_val_probs_sum = {t: np.zeros((len(val_df), len(LABELS[t])), dtype=np.float32) for t in TASKS}
ckip_test_probs_sum = {t: np.zeros((len(test_df), len(LABELS[t])), dtype=np.float32) for t in TASKS}

fold_scores = []
fold_paths = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, strat_key), start=1):
    fold_train_df = train_df.iloc[tr_idx].reset_index(drop=True)
    fold_valid_df = train_df.iloc[va_idx].reset_index(drop=True)

    model, best_score, best_path = train_one_fold(fold, fold_train_df, fold_valid_df, tokenizer)
    fold_scores.append(best_score)
    fold_paths.append(best_path)

    fold_valid_probs = predict_probs(model, fold_valid_df, tokenizer, batch_size=16)
    for t in TASKS:
        ckip_oof_probs[t][va_idx] = fold_valid_probs[t]

    val_probs = predict_probs(model, val_df, tokenizer, batch_size=16)
    for t in TASKS:
        ckip_val_probs_sum[t] += val_probs[t] / N_SPLITS

    test_probs = predict_probs(model, test_df, tokenizer, batch_size=16)
    for t in TASKS:
        ckip_test_probs_sum[t] += test_probs[t] / N_SPLITS

    del model
    torch.cuda.empty_cache()

ckip_obj = {
    "model_name": MODEL_NAME,
    "tasks": TASKS,
    "labels": LABELS,
    "train_path": TRAIN_PATH,
    "val_path": VAL_PATH,
    "test_path": TEST_PATH,
    "fold_paths": fold_paths,
    "fold_scores": fold_scores,
    "oof_probs": ckip_oof_probs,
    "val_probs": ckip_val_probs_sum,
    "test_probs": ckip_test_probs_sum,
    "train_ids": train_df["id"].tolist(),
    "val_ids": val_df["id"].tolist(),
    "test_ids": test_df["id"].tolist(),
}

ckip_prob_path = f"{OUT_DIR}/stage2_ckip_oof_val_test_probs.pkl"
with open(ckip_prob_path, "wb") as f:
    pickle.dump(ckip_obj, f)

print("CKIP fold_scores:", fold_scores)
print("CKIP mean fold score:", np.mean(fold_scores))
print("saved:", ckip_prob_path)


## 6. 單模型基準

In [ ]:

# 如果 runtime 中斷，可以從這裡開始，取消下面註解
# with open(f"{OUT_DIR}/stage2_ckip_oof_val_test_probs.pkl", "rb") as f:
#     ckip_obj = pickle.load(f)
# ckip_val_probs_sum = ckip_obj["val_probs"]
# ckip_test_probs_sum = ckip_obj["test_probs"]
# fold_scores = ckip_obj.get("fold_scores", [])

def eval_single(name, probs, config=None):
    pred = probs_to_pred_df(val_df, probs, config=config or {})
    metrics = calc_metrics(val_df, pred)
    print_metrics(metrics, name)
    for t in TASKS:
        print(t, pred[t].value_counts().to_dict())
    return metrics, pred

mac_raw_metrics, mac_raw_pred = eval_single("MacBERT raw v30", mac_val_probs, {})
mac_cal_metrics, mac_cal_pred = eval_single("MacBERT Stage A2 calibrated", mac_val_probs, V30_01_BEST_CONFIG)
ckip_raw_metrics, ckip_raw_pred = eval_single("CKIP raw Stage 2B", ckip_val_probs_sum, {})


## 7. MacBERT + CKIP Task-wise Ensemble

In [ ]:

def combine_probs(mac_probs, ckip_probs, weights):
    # weights[t] = MacBERT weight; CKIP weight = 1 - weights[t]
    out = {}
    for t in TASKS:
        w = weights.get(t, 0.5)
        out[t] = w * mac_probs[t] + (1 - w) * ckip_probs[t]
    return out

def eval_ensemble_config(config, verbose=False, title=""):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    post_cfg = {k: v for k, v in config.items() if k != "weights"}
    probs = combine_probs(mac_val_probs, ckip_val_probs_sum, weights)
    pred = probs_to_pred_df(val_df, probs, post_cfg)
    metrics = calc_metrics(val_df, pred)
    if verbose:
        print_metrics(metrics, title or "ensemble config")
        print("weights:", weights)
        print("post_cfg:", json.dumps(post_cfg, ensure_ascii=False, indent=2))
        for t in TASKS:
            print(t, pred[t].value_counts().to_dict())
    return metrics["weighted_score"], metrics, pred

def copy_config(cfg):
    return json.loads(json.dumps(cfg))

candidate_starts = []

cfg = copy_config(V30_01_BEST_CONFIG)
cfg["weights"] = {t: 1.0 for t in TASKS}
candidate_starts.append(("macbert_stage_2A", cfg))

cfg = {"weights": {t: 0.0 for t in TASKS}}
candidate_starts.append(("ckip_raw", cfg))

cfg = copy_config(V30_01_BEST_CONFIG)
cfg["weights"] = {t: 0.5 for t in TASKS}
candidate_starts.append(("avg_with_stage_2A_cfg", cfg))

best_start = None
for name, cfg in candidate_starts:
    score, metrics, pred = eval_ensemble_config(cfg, verbose=True, title=f"start: {name}")
    if best_start is None or score > best_start[0]:
        best_start = (score, cfg, metrics)

print("best start:", best_start[0])
print(json.dumps(best_start[1], ensure_ascii=False, indent=2))


In [ ]:

def coordinate_search(base_config, search_space, rounds=4):
    best_cfg = copy_config(base_config)
    best_score, best_metrics, best_pred = eval_ensemble_config(best_cfg)
    history = []

    for r in range(rounds):
        improved = False
        print(f"\n--- coordinate round {r+1} start score {best_score:.6f} ---")

        for key, values in search_space:
            local_best = (best_score, copy_config(best_cfg), best_metrics)
            for val in values:
                cfg = copy_config(best_cfg)

                if key[0] == "weight":
                    _, task = key
                    cfg.setdefault("weights", {})[task] = val
                elif key[0] == "mult":
                    _, task, label = key
                    cfg.setdefault("multipliers", {}).setdefault(task, {})[label] = val
                elif key[0] == "threshold":
                    _, name = key
                    cfg[name] = val
                elif key[0] == "flag":
                    _, name = key
                    cfg[name] = val

                score, metrics, _ = eval_ensemble_config(cfg)
                if score > local_best[0]:
                    local_best = (score, cfg, metrics)

            if local_best[0] > best_score + 1e-9:
                best_score, best_cfg, best_metrics = local_best
                improved = True
                print("improved", key, "->", best_score)
                print(json.dumps(best_cfg, ensure_ascii=False, indent=2))

        history.append({"round": r+1, "score": best_score, "config": copy_config(best_cfg)})
        if not improved:
            break

    best_score, best_metrics, best_pred = eval_ensemble_config(best_cfg)
    return best_cfg, best_score, best_metrics, best_pred, history

weight_values = [round(x, 2) for x in np.arange(0.0, 1.01, 0.05)]
mult_values = [0.10, 0.20, 0.35, 0.50, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0, 6.0]
threshold_values = [round(x, 2) for x in np.arange(0.30, 0.81, 0.02)]

search_space = []

for t in TASKS:
    search_space.append((("weight", t), weight_values))

for lab in LABELS["evidence_quality"]:
    search_space.append((("mult", "evidence_quality", lab), mult_values))

for lab in LABELS["verification_timeline"]:
    search_space.append((("mult", "verification_timeline", lab), mult_values))

search_space.append((("threshold", "t1_threshold"), [None] + threshold_values))
search_space.append((("threshold", "t3_threshold"), [None] + threshold_values))
search_space.append((("flag", "force_quality_non_na_when_evidence_yes"), [False, True]))

best_cfg, best_score, best_metrics, best_pred, history = coordinate_search(
    best_start[1],
    search_space,
    rounds=4,
)

print_metrics(best_metrics, "BEST stage2 ensemble val1000")
print("best config:")
print(json.dumps(best_cfg, ensure_ascii=False, indent=2))
for t in TASKS:
    print("\n", t)
    print(best_pred[t].value_counts().to_dict())


## 8. 輸出 val1000 分析

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )
    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]
    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

best_val_full = save_val_outputs(best_pred, best_metrics, "stage2_best_val")
display(best_val_full["any_error"].value_counts())
display(best_val_full.head())

with open(f"{OUT_DIR}/stage2_best_val_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_config": best_cfg,
        "best_metrics": best_metrics,
        "history": history,
        "mac_raw_metrics": mac_raw_metrics,
        "mac_v30_01_metrics": mac_cal_metrics,
        "ckip_raw_metrics": ckip_raw_metrics,
        "ckip_fold_scores": fold_scores,
    }, f, ensure_ascii=False, indent=2)

print("saved val outputs and config to", OUT_DIR)


## 9. 產生 best / conservative / balanced 三版 submission

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_test_submission(config, tag):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    post_cfg = {k: v for k, v in config.items() if k != "weights"}

    probs = combine_probs(mac_test_probs, ckip_test_probs_sum, weights)
    pred = probs_to_pred_df(test_df, probs, post_cfg)

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=post_cfg.get("force_quality_non_na_when_evidence_yes", False),
        probs=adjust_probs(probs, post_cfg.get("multipliers", {})),
    )

    sub = pred[["id"] + TASKS].copy()
    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

best_sub, best_path = make_test_submission(best_cfg, "stage2_best_val")

conservative_cfg = copy_config(best_cfg)
if conservative_cfg.get("t1_threshold") is not None:
    conservative_cfg["t1_threshold"] = min(float(conservative_cfg["t1_threshold"]) + 0.04, 0.80)
if conservative_cfg.get("t3_threshold") is not None:
    conservative_cfg["t3_threshold"] = min(float(conservative_cfg["t3_threshold"]) + 0.04, 0.80)
conservative_sub, conservative_path = make_test_submission(conservative_cfg, "stage2_conservative")

balanced_cfg = copy_config(best_cfg)
balanced_cfg.setdefault("multipliers", {})
if "verification_timeline" in balanced_cfg["multipliers"]:
    if "within_2_years" in balanced_cfg["multipliers"]["verification_timeline"]:
        balanced_cfg["multipliers"]["verification_timeline"]["within_2_years"] = max(
            1.0, float(balanced_cfg["multipliers"]["verification_timeline"]["within_2_years"]) * 0.75
        )
if "evidence_quality" in balanced_cfg["multipliers"]:
    if "Clear" in balanced_cfg["multipliers"]["evidence_quality"]:
        balanced_cfg["multipliers"]["evidence_quality"]["Clear"] = max(
            1.0, float(balanced_cfg["multipliers"]["evidence_quality"]["Clear"]) * 0.85
        )
balanced_sub, balanced_path = make_test_submission(balanced_cfg, "stage2_balanced")

print("\nSubmission paths:")
print(best_path)
print(conservative_path)
print(balanced_path)


## 10. 報告摘要

In [ ]:

report = []
report.append("# Stage 2 CKIP + MacBERT 任務別集成報告")
report.append("")
report.append("## Single model baselines")
report.append("### MacBERT raw stage1")
for k, v in mac_raw_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("### MacBERT stage1 calibrated")
for k, v in mac_cal_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("### CKIP raw stage2")
for k, v in ckip_raw_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## Best stage2 ensemble")
for k, v in best_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## Best config")
report.append("```json")
report.append(json.dumps(best_cfg, ensure_ascii=False, indent=2))
report.append("```")

report.append("")
report.append("## Test distribution: stage2_best_val")
for t in TASKS:
    report.append(f"### {t}")
    for lab, cnt in best_sub[t].value_counts().to_dict().items():
        report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)
with open(f"{OUT_DIR}/stage2_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:5000])
print("saved:", f"{OUT_DIR}/stage2_report.md")
